# Jev calibration for semantic `ORDER BY`Independent calibration measurements for TypeSafe AI's Jev model, run aspart of `jev-orderby-bench`. No independent ranking numbers exist publicly; thevendor's own eval reports 67.8% agreement against averaged frontierjudgments, self-run and unreproduced. Agreement with other models is notcalibration, so nothing here is compared against it.**Why calibration comes first.** The product is `ORDER BY` over a semanticscore. If the probabilities are not calibrated, the sort key is ameaningless number and every query fails silently: the rows come back in*an* order, just not a defensible one. That failure is invisible intesting, which is why this notebook gates Phase 2.

## What is measured, and why two families**Calibration** (Brier, ECE) asks: is a stated 0.7 really 70%?**Ranking** (Spearman, pairwise inversion, AUC) asks: does sorting by thisput rows in the right order?These come apart in both directions, which is why a gate on ECE alone isthe wrong gate:- Probabilities squashed into [0.48, 0.52] but perfectly ordered: awful  ECE, flawless sort. (The test suite demonstrates this case: ECE 0.485,  zero inversions.)- Well calibrated in aggregate but many inverted pairs: good ECE, visibly  wrong page of results.`ORDER BY` depends on the second family. The published spec's gate namedonly the first.

In [ ]:
import json, sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / "harness"))import numpy as npimport metrics as Mimport run_calibration as RRESULTS = Path.cwd().parent / "results" / "results.json"report = json.loads(RESULTS.read_text()) if RESULTS.exists() else Noneprint("results found" if report else      "No results yet. Run:  python3 harness/run_calibration.py")

## Corpus and label provenanceThis is the part that decides whether the numbers mean anything.**The labels are not ours.** The corpus is 20 Newsgroups, and eachdocument's label is the newsgroup its author chose to post it to. That isa human judgment recorded by a human at the time, independent of thisproject. Had we hand-labeled the corpus ourselves, we would be measuringJev's agreement with Claude and publishing it as calibration: anauthoritative-looking number worth nothing.Selection constraints, in the order they bound the result:1. Human-labeled by provenance, freely redistributed for research (no explicit license; see README).2. Single-factor labels, matching the spec's rule for questions.3. Clear of Jev's documented weak spots. The `jev-1.13` jaggedness page   names math/counting, date comparison and hex/RGB representations.   Topic membership touches none, so miscalibration we measure is not   secretly a counting failure.4. **Labels span the probability range.** The subtle one. Sampling only   obvious cases pins predictions at 0 and 1, ECE comes out tiny, the   gate passes vacuously and we learn nothing.Constraint 4 drives the stratified sampler: every probe draws clearpositives, topically adjacent **near misses**, and plainly unrelatednegatives. The near-miss rows carry the calibration information; a modelthat is confident there is confidently wrong somewhere.

### The negative labels are weaker than the positive onesThe positives are safe: the author chose `sci.med`, so "is this aboutmedicine" is yes. The **negatives are not all sound**. A`talk.politics.misc` post about healthcare reform genuinely is abouthealth; a `rec.autos` post selling a part genuinely is offering an itemfor sale. The author picking a different newsgroup does not entail"not about X."That is sharper than generic label noise, because the samplerconcentrates the least reliable labels in the largest stratum *and* inthe mid-probability region where ECE is decided. A correct 0.6 scoredagainst a wrong `False` reads as miscalibration and could fail the gateon label error rather than model error.So doubtful rows are detected **per row, not per group**, flagged`label_confident=False` (7 of 360), excluded from every *gated*calibration metric, and kept for ranking (ranking only uses pairs thelabels do order, and those hard rows are where sort order matters most).The all-rows figure is reported beside each gated one so the effect ofthe exclusion is visible rather than hidden.Banning whole groups was the first attempt and backfired: contaminationis only 2-5% per group, and it removed both of `forsale`'s near-missgroups, collapsing that probe to clear positives plus clear negatives.That is the vacuous-gate failure constraint 4 exists to prevent. Per-rowfiltering keeps all three strata on all three probes.

In [ ]:
rows = R.load_corpus()from collections import Counterprint(f"{len(rows)} rows")print("probe   :", dict(Counter(r["probe"] for r in rows)))print("stratum :", dict(Counter(r["stratum"] for r in rows)))print("label   :", dict(Counter(r["label"] for r in rows)))print()ex = next(r for r in rows if r["stratum"] == "near_miss")print(f"near-miss example [{ex['newsgroup']}] label={ex['label']}")print(ex["text"][:300])

## Boolean (`jev_bool` / Noul)Brier with the Murphy decomposition, and ECE on equal-mass bins.Two choices worth stating. **Adaptive binning**: with ~360 rows, fixedwidth bins leave some nearly empty, and an empty bin's error is noise thatstill gets weighted into the total. **Wilson intervals** on every bin,because a reliability diagram without error bars invites reading structureinto sampling noise.The decomposition separates two failures the single Brier number hides:high *reliability* means the probabilities are wrong; low *resolution*means they are uninformative. A model that predicts the base rate forevery row scores a respectable Brier and is useless for `ORDER BY`, soresolution is a gate condition in its own right.

In [ ]:
if report and "boolean" in report:    b = report["boolean"]    print(f"n = {b['n']}  (calibration on {b['n_calibration']}, "          f"{b['n_excluded_ambiguous']} ambiguous held out)")    print(f"ECE incl. ambiguous rows: {b['ece_all_rows_incl_ambiguous']:.4f}")    print(f"Brier            {b['brier']:.4f}")    d = b["decomposition"]    print(f"  reliability    {d['reliability']:.4f}  (lower better)")    print(f"  resolution     {d['resolution']:.4f}  (higher better)")    print(f"  uncertainty    {d['uncertainty']:.4f}  (base rate)")    e = b["ece"]    print(f"ECE ({e['binning']}, {e['n_bins_used']} bins)  {e['ece']:.4f}")    print(f"ECE (fixed bins)     {b['ece_fixed_bins']:.4f}")    print(f"MCE (worst bin)      {e['mce']:.4f}")    r = b["ranking"]    print(f"\nRanking: AUC {r.get('auc', float('nan')):.4f}  "          f"Spearman {r['spearman']:.4f}  inversions {r['inversion_rate']:.4f}")else:    print("awaiting a scored run")

In [ ]:
if report and "boolean" in report:    print(f"{'n':>5} {'pred':>7} {'obs':>7} {'gap':>7}   95% CI")    for bn in report["boolean"]["ece"]["bins"]:        print(f"{bn['n']:>5} {bn['mean_pred']:>7.3f} {bn['observed']:>7.3f} "              f"{bn['gap']:>7.3f}   [{bn['ci_low']:.3f}, {bn['ci_high']:.3f}]")

### Reliability diagram

In [ ]:
if report and "boolean" in report:    from IPython.display import Image, display    p = Path.cwd().parent / "results" / "reliability.png"    display(Image(str(p))) if p.exists() else print("no diagram yet")

## The negation invariantThe strongest measurement available here, and the one that needs **noground-truth labels at all**.The `jev-1.13` jaggedness page states there is no guarantee that`P(noul)` equals `1 - P(not noul)` across separate questions. Thatdisclaimer makes the violation worth measuring rather than assuming away.Because it compares the model against itself, it is completely immune tothe label-provenance problem that makes any hand-labeled calibrationnumber suspect. A large asymmetry means the probability moves withquestion phrasing as much as with evidence, which undermines any thresholda caller sets and therefore any `WHERE prob > x` clause.Both framings are asked in the same request, against identical state.

In [ ]:
if report and "negation_invariant" in report:    n = report["negation_invariant"]    print(f"n = {n['n']} paired framings")    print(f"mean   |P(q) + P(not q) - 1|   {n['mean_abs_violation']:.4f}")    print(f"median                         {n['median_abs_violation']:.4f}")    print(f"p95                            {n['p95_abs_violation']:.4f}")    print(f"max                            {n['max_abs_violation']:.4f}")    print(f"signed bias                    {n['signed_bias']:+.4f}"          "   (>0 = leans yes to both framings)")    print(f"fraction violating > 0.10      {n['frac_over_0.10']:.1%}")    print(f"fraction violating > 0.20      {n['frac_over_0.20']:.1%}")else:    print("awaiting a scored run")

### The confound, and the controlA large asymmetry has two possible explanations: Jev violates theidentity, or the two strings were never true logical complements. Toseparate them:1. The negation is derived **mechanically** from the positive question   rather than hand-written, so the only difference between framings is   the inserted negation.2. A **paraphrase control** asks a semantically equivalent reworded   question. It should agree with the positive, so the disagreement it   shows is the floor for how much wording alone moves this model.If negation disagreement is not clearly larger than paraphrasedisagreement, the result is general wording sensitivity and must bereported as such, not as a finding about negation.

In [ ]:
if report and "paraphrase_control" in report:    pc = report["paraphrase_control"]    neg = report.get("negation_invariant", {})    print(f"paraphrase mean |diff|   {pc['mean_abs_diff']:.4f}   (control: should be ~0)")    print(f"negation   mean |viol|   {neg.get('mean_abs_violation', float('nan')):.4f}")    r = pc.get("negation_to_paraphrase_ratio")    if r is not None:        print(f"\nratio {r:.2f}x")        print("negation-specific" if r > 2 else              "NOT negation-specific: wording sensitivity explains most of it")else:    print("awaiting a scored run")

## ChoiceFor Choice, calibration means: does the stated `confidence` predictwhether the pick was actually right? So Brier and ECE are computed onconfidence against correctness. An `other` option is included so the modelcan decline rather than being forced into a wrong bucket.

In [ ]:
if report and "choice" in report:    c = report["choice"]    print(f"n {c['n_calibration']} confident of {c['n_all_rows']} scored")    print(f"accuracy             {c['accuracy']:.4f}"          f"   (all rows: {c['accuracy_all_rows']:.4f})")    print(f"Brier on confidence  {c['brier_on_confidence']:.4f}")    print(f"ECE on confidence    {c['ece_on_confidence']['ece']:.4f}   <- gated")    print(f"ECE incl. ambiguous  {c['ece_all_rows_incl_ambiguous']:.4f}")    print(f"resolution           {c['decomposition']['resolution']:.4f}")else:    print("awaiting a scored run")

## Score**Brier and ECE are omitted here on purpose.** They are classificationmetrics; they do not apply to a continuous score. Reporting them wouldproduce a number that looks comparable to the Boolean row and is not.**The scale matters and is easy to get wrong.** Score returns aprobability-weighted mean over level *indices*, so an `n`-level rubricspans `0..n-1`, not `0..1`. Two consequences for SQL:- `jev_score_val` output is **not comparable across different rubrics**.- `ORDER BY` mixing rubrics is meaningless, and nothing in SQL will warn.The stratum check is the ordinality test: levels are evaluatedindependently and the model never sees level numbers, so ordinality isimposed entirely by *our* array order. If mean score does not increasefar < near_miss < positive, the rubric is not actually ordinal and everysort key built from it is noise.

In [ ]:
if report and "score" in report:    s = report["score"]    sc = report["score_scale"]    print(f"n {s['n']}   scale 0..{sc['range'][1]} ({sc['levels']} levels)")    print(f"observed range {s['observed_range']}   mean {s['mean']:.3f}")    r = s["ranking_vs_label"]    print(f"\nSpearman {r['spearman']:.4f}   Kendall {r['kendall_tau']:.4f}")    print(f"inversion rate {r['inversion_rate']:.4f} "          f"over {r['comparable_pairs']:,} comparable pairs")    print(f"\nstratum means: {s.get('stratum_means')}")    print(f"monotonic (far < near < positive): {s.get('stratum_monotonic')}")else:    print("awaiting a scored run")

### Rubric ordinalityThe check that our array order corresponds to something the modelactually perceives as ordered. The same rubric is scored a second time**reversed**; a genuinely ordinal scale must mirror:`score_reversed ≈ scale_max - score`.This needs no ground truth, and it is load-bearing: levels are scoredindependently and the model never sees level numbers, so if the mirrorfails, the ordering is our assumption rather than the model's, and every`ORDER BY jev_score_val(...)` is sorting noise.

In [ ]:
if report and "rubric_ordinality" in report.get("score", {}):    ro = report["score"]["rubric_ordinality"]    print(f"n {ro['n']}")    print(f"mean |score - (max - score_reversed)|   {ro['mean_abs_mirror_error']:.4f}")    print(f"median                                  {ro['median_abs_mirror_error']:.4f}")    print(f"p95                                     {ro['p95_abs_mirror_error']:.4f}")    print(f"as fraction of full scale               {ro['mean_error_as_scale_fraction']:.1%}")    print(f"correlation with mirrored               {ro['correlation_with_mirror']:.4f}")else:    print("awaiting a scored run")

## GateThresholds were fixed **before** any results were seen, so they cannot berationalised afterwards. The three primitives are gated separately, notjust reported separately: gating only `jev_bool` would let a Score thatinverts a third of its pairs through, and `jev_score_val` is the accessor`ORDER BY` actually sorts on.| Condition | Threshold | Why ||---|---|---|| `jev_bool` ECE | ≤ 0.10 | A stated 0.8 that is really 0.7 is tolerable for ranking; wider and the probability is decorative. || `jev_bool` inversion | ≤ 0.15 | Past roughly one bad pair in six, a sorted page looks visibly wrong. || `jev_bool` resolution | > 0 | At or below zero the model is not separating classes at all. || **`jev_score` inversion** | ≤ 0.15 | Graded ranking vs the 3-level ordinal stratum, so it sees mis-ordering *within* the positives. Necessary condition for semantic ORDER BY. || `jev_choice` confidence ECE | ≤ 0.10 | Does stated confidence predict a correct pick? || Negation asymmetry | ≤ 0.15 | Beyond this, phrasing moves the answer as much as evidence does. |If the gate fails, the spec is explicit: stop and rethink before Phase 2.`udf.register()` enforces this in code and refuses to register the SQLfunctions unless the gate passed.

In [ ]:
if report and "gate" in report:    g = report["gate"]    print("GATE:", "PASS" if g["passed"] else "FAIL")    for f in g["failures"]:        print("  -", f)    print("\nthresholds:", json.dumps(g["thresholds"], indent=2))    u = report.get("usage", {})    print(f"\ncost: {u.get('requests')} requests, "          f"{u.get('input_tokens',0):,} in + {u.get('output_tokens',0):,} out tokens")else:    print("awaiting a scored run")

## LimitationsStated plainly, because the point of the artifact is to be trustworthy.- **One corpus, one domain.** English newsgroup posts. Calibration is a  property of model *and* domain; these numbers do not transfer to  contracts, support tickets, or governance proposals without re-running.- **Topic membership is an easy judgment.** Real ORDER BY workloads ask  harder questions, so treat these as an upper bound.- **~120 rows per probe.** Ten-bin ECE is noisy at that size, which is why  bins carry Wilson intervals and adaptive binning is the default. Read  the intervals, not the third decimal.- **The 20 Newsgroups labels are noisy.** Authors cross-post and pick  imperfect groups, so some ground truth is wrong. That inflates apparent  miscalibration.- **Invariants are necessary, not sufficient.** Passing negation symmetry  does not imply calibration. Failing it does imply the probabilities  cannot be thresholded reliably.- **`jev-latest` is a moving target.** Record the `model` field from  responses; these numbers attach to one version.